<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.13 Local Agents I

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

Every model you have run in this course had the same strange disability: it could *describe* the world, but not *touch* it. Ask `qwen3:1.7b` about the Wi-Fi in your building and it will produce confident, fluent and entirely invented numbers, because generating plausible text is all a language model can do. Today we give a model on your Raspberry Pi **tools**, real functions it can ask to have executed and then wrap it in a **loop** so it can pursue a goal across several steps. That combination has a name:

$$
\textbf{agent} \;=\; \text{model} \;+\; \text{tools} \;+\; \text{loop}
$$

It is worth being precise here, because the word "agent" is used loosely in the industry. A **chatbot** answers and stops. A **workflow** is a fixed sequence of model calls that *you* designed. An **agent** is different in kind: the model itself decides, at every step, what to do next: which tool to call, with which arguments, and when it is done [02]. That autonomy is what makes agents powerful, and it is exactly what makes them expensive and risky, which is why this notebook lets you watch one work step by step rather than just talk about it.

Running the agent *locally*, on your own Pi, is not just course convenience. An agent sees everything its tools can see, and with a local model, that evidence never leaves your hardware. Recall the deployment reasoning from [`05.07_Introduction_to_Local_LLMs.ipynb`](./05.07_Introduction_to_Local_LLMs.ipynb): for an *agent*, which acts rather than talks, those arguments carry even more weight.

Before starting, make sure you have worked through [`05.04_GettingStarted_with_Raspberry_Pi.ipynb`](./05.04_GettingStarted_with_Raspberry_Pi.ipynb): this notebook talks to the Pi from Python, so the device has to be connected.

### 2. The Phone Line: Reaching the Pi from Python

Until now you talked to Ollama by typing into an SSH session. An agent cannot type. Our notebook needs to reach the Pi's API **programmatically**. The clean way is an **SSH tunnel**: one command that makes the Pi's Ollama appear as a local port on your laptop, with everything traveling encrypted over the Ethernet link you set up in the first Pi session.

Open a *separate* terminal, run the following, and leave it open for the rest of this notebook:

```sh
ssh -L 11434:localhost:11434 cas@172.31.250.1
```

```text
 your laptop                                Raspberry Pi
+--------------------------+     SSH      +--------------------------+
|  this notebook           |   tunnel     |  Ollama                  |
|  localhost:11434  o======+==============+=====o  localhost:11434   |
+--------------------------+  (encrypted  +--------------------------+
                              port forward)
```

Read the command as: *"forward my local port 11434 through the SSH connection to port 11434 on the Pi."* Every request the notebook sends to `localhost:11434` pops out on the Pi as if it had been made there. This trick (tunneling a service through an encrypted connection) is everyday networking practice, and you will reuse it in [`05.14_Local_Agents_II.ipynb`](./05.14_Local_Agents_II.ipynb) to plug the Pi into VS Code.

In [ ]:
import requests

# The Pi's Ollama API, seen through the SSH tunnel (keep that terminal open).
OLLAMA_URL = "http://localhost:11434"

try:
    version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json()["version"]
    models = sorted(m["name"] for m in
                    requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"])
    PI_ONLINE = True
    print(f"✅ connected to Ollama {version} through the tunnel")
    print(f"   models available: {', '.join(models)}")
except requests.exceptions.RequestException:
    PI_ONLINE = False
    print(f"❌ nothing answered at {OLLAMA_URL}")
    print("   Connect the Pi, open the tunnel (see above), then rerun this cell.")

### 3. Tool Calling: Giving the Model Hands

A **tool** is an ordinary function on *your* side, plus a description of it, a **schema**, that we hand to the model along with the conversation. One point is worth stating early because it simplifies everything that follows:

> **The model never executes anything.** It can only do what it has always done: emit text. "Calling a tool" means the model emits a small, structured request (*"please run `measure_wifi_signal` with `room = kitchen`"*). Your code decides whether to actually run the function, runs it, and sends the result back as a new message. The model has hands only because your loop lends it yours.

Our tools measure something this course cares about: Wi-Fi signal strength in dBm, per room. (The measurements are a fixed table standing in for a real site survey; the mechanics are identical.) Note that `gemma3` does not support tool calling, so we use `qwen3:1.7b`, with its thinking mode off as in [`05.12_Understanding_Reasoning_Models.ipynb`](./05.12_Understanding_Reasoning_Models.ipynb).

We first make a **single** tool round trip, with every message printed, so you can see the exact mechanism before we automate it.

In [ ]:
AGENT_MODEL = "qwen3:1.7b"     # tool-capable and small; gemma3 cannot call tools

# --- the real functions (a fixed site survey standing in for live measurements) ---
ROOM_SIGNALS_DBM = {"office": -48, "kitchen": -63, "basement": -81, "garden": -71}

def list_rooms():
    return list(ROOM_SIGNALS_DBM)

def measure_wifi_signal(room):
    return f"{ROOM_SIGNALS_DBM.get(room.lower().strip(), -95)} dBm"

TOOL_FUNCTIONS = {"list_rooms": list_rooms, "measure_wifi_signal": measure_wifi_signal}

# --- the schemas the model gets to see ---
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "list_rooms",
        "description": "List the rooms of the building.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "measure_wifi_signal",
        "description": "Measure the current Wi-Fi signal strength in one room, in dBm.",
        "parameters": {"type": "object",
                       "properties": {"room": {"type": "string",
                                               "description": "name of the room to measure"}},
                       "required": ["room"]}}},
]

def chat(messages):
    """One /api/chat request with our tools attached."""
    reply = requests.post(f"{OLLAMA_URL}/api/chat", json={
        "model": AGENT_MODEL, "messages": messages, "tools": TOOL_SCHEMAS,
        "stream": False, "think": False,
        "options": {"temperature": 0.2, "seed": 7},
    }, timeout=600).json()
    return reply["message"]

if PI_ONLINE:
    messages = [{"role": "user", "content": "How strong is the wi-fi in the kitchen?"}]

    reply = chat(messages)
    if not reply.get("tool_calls"):
        print("the model answered directly this time (small models occasionally skip the tool):")
        print("  ", reply["content"].strip())
        print("rerun this cell until you catch it calling the tool.")
    else:
        print("the model's reply is not text - it is a request:")
        print("  ", reply.get("tool_calls"))

        call = reply["tool_calls"][0]["function"]
        result = TOOL_FUNCTIONS[call["name"]](**(call.get("arguments") or {}))
        print(f"\nour code executes {call['name']}({call.get('arguments')}) -> {result}")

        messages += [reply, {"role": "tool", "content": str(result), "tool_name": call["name"]}]
        print("\nfinal answer, grounded in the measurement:")
        print("  ", chat(messages)["content"].strip())
else:
    print("❌ Pi not connected - rerun the connection cell first.")

Look closely at what was printed. The model's first reply contained no prose at all: just a machine-readable request naming a function *from the schema we provided* and arguments *it extracted from the question*. Our three lines of Python did the actual work, and only then, with the measurement in the conversation, did the model produce its answer. Compare that with what the model would have said without tools: a fluent guess, as in the `satellite` demo of [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb).

### 4. The Agent Loop

One round trip is tool calling. An **agent** emerges when we put it in a loop: the model keeps choosing tools, seeing results, and deciding what to do next, until it declares itself done or hits a **step budget**. This think-act-observe cycle is the ReAct pattern [01], and it is the engine inside every agent product you have read about, from coding assistants to computer-use agents. The loop below differs from them in scale, not in kind.

The step budget matters more than it looks. An agent that cannot finish will happily loop forever and every step is a full LLM inference: on the Pi several seconds, in a company deployment real money. `max_steps` is your first **guardrail** and it will not be the last: notice that in the code below, *your* loop remains the place where every tool request could be inspected, logged, or refused before execution. Keep that in mind for the questions.

In [ ]:
import json

def run_agent(task, max_steps=6):
    """model + tools + loop: keep going until the model answers or the budget runs out."""
    if not PI_ONLINE:
        print("❌ Pi not connected - rerun the connection cell first.")
        return
    messages = [
        {"role": "system", "content":
         "You are a facilities assistant. Gather evidence with the available tools "
         "before answering. Keep the final answer to at most two sentences."},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        reply = chat(messages)
        messages.append(reply)

        if not reply.get("tool_calls"):                       # no request -> the agent is done
            print(f"[step {step}] ANSWER: {reply['content'].strip()}")
            return

        for call in reply["tool_calls"]:                      # execute what the agent asked for
            name = call["function"]["name"]
            arguments = call["function"].get("arguments") or {}
            if isinstance(arguments, str):                    # some versions deliver JSON text
                arguments = json.loads(arguments)
            try:
                result = TOOL_FUNCTIONS[name](**arguments)
            except (KeyError, TypeError) as error:            # feed mistakes back, don't crash
                result = f"tool error: {error}"
            print(f"[step {step}] TOOL   {name}({arguments}) -> {result}")
            messages.append({"role": "tool", "content": str(result), "tool_name": name})

    print(f"[stopped] step budget of {max_steps} reached - the guardrail ended the run.")


run_agent("Which room is best for an important video call? "
          "Check the signal in every room before you decide.")

### 5. Reading the Trace

The printed trace is the deliverable of this notebook. Read it like a lab protocol. A good run shows a plan unfolding: discover the rooms, measure each one, then commit to an answer grounded in the numbers (the office, at −48 dBm, should win comfortably. Recall from [`02.7_Path_Loss_of_Radio_Communication.ipynb`](../02_Data_Communication/02.7_Path_Loss_of_Radio_Communication.ipynb) that less negative dBm means more received power). A bad run is *more* instructive: small models skip rooms, invent room names, call the same tool twice, or answer before measuring anything. Rerun the cell a few times and collect specimens.

Whatever you observe, three structural risks distinguish agents from chatbots, and they should shape how much autonomy you would grant one in production:

- **Compounding errors:** a chatbot's mistake costs one bad answer. An agent's early mistake feeds every later step. Ten steps at 95 % reliability each are a coin flip overall ($0.95^{10} \approx 0.60$).
- **Cost and latency multiply:** every loop iteration is a full inference plus a growing context. Agents are the most expensive way to use a model, worth it only when the task genuinely needs multiple autonomous steps [02].
- **Actions are irreversible in a way words are not:** our tools only read a table. The moment a tool can *write*, the loop needs permissions, logging, and human confirmation for consequential steps. The guardrail belongs in the loop code, on your side of the line, precisely because that is the one place the model cannot talk its way past.

The first risk deserves a picture, because its scale surprises everyone at first contact. The chart below is pure arithmetic (per-step reliability raised to the number of steps) and it is the most important capacity-planning fact about agents:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

steps = np.arange(1, 21)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
for per_step, color in [(0.99, "#00441b"), (0.95, "#238b45"), (0.90, "#74c476")]:
    ax.plot(steps, per_step ** steps, color=color, linewidth=1.8, marker="o", markersize=4)
    ax.annotate(f"{per_step:.0%} per step", (steps[-1], per_step ** steps[-1]),
                textcoords="offset points", xytext=(8, 0), va="center", color=color, fontsize=9)

ax.axhline(0.5, color="#757575", linestyle=":", linewidth=1)
ax.text(1.3, 0.52, "coin flip", color="#757575", fontsize=9)
ax.scatter([10], [0.95 ** 10], s=70, color="#238b45", zorder=3)
ax.annotate(f"10 steps at 95 %: {0.95 ** 10:.0%}", (10, 0.95 ** 10),
            textcoords="offset points", xytext=(10, 12), color="#238b45", fontsize=9)

ax.set_xlabel("steps in the agent run")
ax.set_ylabel("probability the whole run is flawless")
ax.set_xlim(1, 24)
ax.set_ylim(0, 1.05)
ax.set_title("Compounding: reliability is exponential in the number of steps", loc="left")
plt.show()

Read your own agent against this chart: a four-step run at 95 % per-step reliability still succeeds four times out of five, while a twenty-step run is more likely to fail than not. This curve is why production agents keep runs short, verify intermediate results, and escalate to humans, and why "just let it run longer" is never the plan.

### 6. Summarizing Remarks

An agent is not a new kind of intelligence. It is the same next-token predictor you trained in [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb), handed a schema and a loop. That deflationary view is the most useful thing a decision-maker can carry out of this notebook: when a vendor says *agent*, ask what the tools are, who executes them, what the step budget is, and where a human sits in the loop. You have now built the whole mechanism yourself, on a computer the size of a pack of butter, so none of those questions should feel abstract again.

Two threads continue from here: [`05.14_Local_Agents_II.ipynb`](./05.14_Local_Agents_II.ipynb) puts several models to work as a *team* and turns your Pi into a coding assistant, and [`05.15_Local_Model_Context_Protocol.ipynb`](./05.15_Local_Model_Context_Protocol.ipynb) shows the industry standard that replaces our hand-written tool glue.

### Further Reading and Literature

[01] Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2023). **ReAct: Synergizing Reasoning and Acting in Language Models.** ICLR. [Paper](https://arxiv.org/abs/2210.03629)

[02] Anthropic (2024). **Building Effective Agents.** Engineering blog with a readable taxonomy of workflows versus agents, and when *not* to build one. [Article](https://www.anthropic.com/engineering/building-effective-agents)

[03] Ollama (2024). **Tool support.** How the API used in this notebook works. [Blog](https://ollama.com/blog/tool-support)

[04] Schick, T., Dwivedi-Yu, J., Dessì, R., et al. (2023). **Toolformer: Language Models Can Teach Themselves to Use Tools.** NeurIPS. [Paper](https://arxiv.org/abs/2302.04761)

----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**In the single round trip of Section 3, who executed `measure_wifi_signal`: the model or your notebook? Paste the `tool_calls` structure from your run and explain what the model actually produced.** Answer:

(add your answer here)

**Explain the tunnel command `ssh -L 11434:localhost:11434 cas@172.31.250.1` in your own words: which port lives where, and why does the notebook talk to `localhost` even though the model runs on the Pi?** Answer:

(add your answer here)

**Paste the trace of your agent run. How many steps did it take, which tools did it call in which order, and did it check every room before answering?** Answer:

(add your answer here)

**Rerun the agent two or three times. Did the traces differ? Report one imperfection you observed (skipped room, repeated call, invented name, premature answer), or, if every run was clean, explain what about the task made it easy.** Answer:

(add your answer here)

**Add a fifth room to `ROOM_SIGNALS_DBM` and rerun the agent. Did it discover and measure the new room without any change to the task text? What does that tell you about where the agent's "knowledge" of the building lives?** Answer:

(add your answer here)

**Your team proposes an agent that autonomously executes ten-step procedures on customer accounts. Read the ten-step success probability for the 95 % line off your compounding chart, then name two guardrails from this notebook you would require before approving it, and one metric you would monitor.** Answer:

(add your answer here)

**A vendor demo shows an "AI agent" that summarizes a weekly report on request. Using the chatbot / workflow / agent distinction from Section 1, what would you ask to find out which of the three it really is, and why does the answer matter for the risk assessment?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)